# 01 — Load raw SISAIRE data (Bogotá)

Loads all `BG_*_reporte_sisaire.csv` files (PM2.5, PM10, NO2, O3, wind speed/direction,
temperature at 2m/10m, relative humidity at 2m/10m, pressure) from `Data/COLOMBIA` and
merges them into one tidy `(station, date)` table.

No cleaning happens here — this notebook only loads and merges. QC (Chauvenet's
criterion, completeness thresholds, physical range checks) is done in
`02_quality_control.ipynb`.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import pandas as pd
from common.sisaire_io import load_city_dataset, BOGOTA_VARIABLES

DATA_DIR = PROJECT_ROOT / "Data" / "COLOMBIA"
OUT_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
raw = load_city_dataset(DATA_DIR, city_prefix="BG", variables=BOGOTA_VARIABLES)
print(raw.shape)
raw.head()


(92192, 13)


,station,date,pm25,pm10,no2,o3,wind_speed,wind_dir,temp_2m,temp_10m,rh_2m,rh_10m,pressure
0,BOGOTA RURAL - MOCHUELO,2010-01-04,NaN,59.37,47.69,59.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BOGOTA RURAL - MOCHUELO,2010-01-06,NaN,68.72,42.61,63.57,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BOGOTA RURAL - MOCHUELO,2010-01-10,NaN,72.56,54.83,66.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BOGOTA RURAL - MOCHUELO,2010-01-14,NaN,69.33,57.20,67.24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BOGOTA RURAL - MOCHUELO,2010-01-17,NaN,64.02,48.62,68.62,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
print("Stations:", raw['station'].nunique())
print("Date range:", raw['date'].min(), "to", raw['date'].max())
print()
print("Non-null counts per variable:")
print(raw.drop(columns=['station', 'date']).notna().sum().sort_values(ascending=False))


Stations: 24
Date range: 2010-01-01 00:00:00 to 2026-08-19 00:00:00

Non-null counts per variable:
pm10          83439
pm25          66615
o3            64607
no2           57520
wind_speed    55270
wind_dir      55016
temp_2m       47613
rh_2m         34662
pressure      29244
temp_10m       3712
rh_10m         1486
dtype: int64


In [4]:
raw.groupby('station').size().sort_values(ascending=False)


station
PUENTE ARANDA                 6044
SUBA                          6013
LAS FERIAS                    6004
GUAYMARAL                     6004
CENTRO DE ALTO RENDIMIENTO    6002
USAQUEN                       5959
CARVAJAL - SEVILLANA          5956
KENNEDY                       5926
SAN CRISTOBAL                 5715
TUNAL                         5663
MINAMBIENTE                   5460
BOGOTA RURAL - MOCHUELO       4682
BOLIVIA                       3882
P_CAMI - FONTIBÓN             2788
MÓVIL 7MA                     2713
EL JAZMÍN                     2130
USME                          2130
CIUDAD BOLÍVAR                2115
COLINA                        1922
MÓVIL FONTIBÓN                1908
Fontibon                      1505
MOCHUELO - COLEGIO            1067
BOSA                           323
CHICO LAGO (STO TOMAS)         281
dtype: int64

## Save raw merged table

In [5]:
out_path = OUT_DIR / "bogota_raw_merged.parquet"
raw.to_parquet(out_path, index=False)
print("Saved:", out_path, raw.shape)


Saved: /path/to/project/Data/processed/colombia/bogota/bogota_raw_merged.parquet (92192, 13)
